# Step 13.4 — Transpose for Batched Attention：为什么要从 `(B,T,H,D)` 变成 `(B,H,T,D)`？

Step 13.3 已经把 full-width Q/K/V channels 组织成：

$$
Q_h,K_h,V_h
\in
\mathbb R^{B\times T\times H\times D}.
$$

其中：

```text
B = batch
T = sequence length
H = number of heads
D = head dimension
```

现在虽然 H 已经是显式 axis，但还不适合直接写每个 head 的：

$$
Q^{(h)}K^{(h)T}.
$$

最常见的下一步是把 T 与 H 两个 axes 交换：

$$
\boxed{
(B,T,H,D)
\rightarrow
(B,H,T,D)
}
$$

这样 `(B,H)` 可以作为 batch-like dimensions，而最后两个 axes `(T,D)` 正好就是每个 head 自己的 Query matrix。

本节只做到 batched head-wise attention organization；不进入 Step 13.5 的 head concatenation 或 output projection。


## 1. 为什么 `(B,T,H,D)` 不够方便？

在 `(B,T,H,D)` 中，最后两个 axes 是：

```text
H, D
```

而 MLX batched matrix multiplication 会把**最后两个 axes**解释成真正要做 matrix product 的 matrix dimensions。

但对每个 head，我们真正想算的是：

$$
(T,D)
@
(D,T)
\rightarrow
(T,T).
$$

也就是：

> 每个 head 内部，所有 T 个 Query positions 对所有 T 个 Key positions 做 pairwise dot products。

所以需要让：

```text
最后两个 axes = T, D
```

这正是：

$$
(B,H,T,D)
$$

的布局。

于是前面的：

$$
(B,H)
$$

就自然成为 batched matmul 的 batch dimensions。


## 2. 只交换 T 和 H：`swapaxes(1,2)`

原 tensor：

$$
Q_{BTHD}:(B,T,H,D).
$$

交换 axis 1 与 axis 2：

$$
\boxed{
Q_{BHTD}
=
\operatorname{swapaxes}(Q_{BTHD},1,2)
}
$$

得到：

$$
Q_{BHTD}:(B,H,T,D).
$$

K、V 同理。

这一步没有改变：

- 元素数量；
- learned parameters；
- 每个元素本身的数值。

只是改变 axes 的逻辑顺序。

索引关系可以写成：

$$
\boxed{
Q_{BHTD}[b,h,t,d]
=
Q_{BTHD}[b,t,h,d]
}.
$$

注意：这和 Step 13.3 的 reshape 是两种不同操作：

```text
13.3 reshape
C -> H,D

13.4 axis swap
B,T,H,D -> B,H,T,D
```


In [ ]:
import mlx.core as mx

B, T, H, D = 1, 3, 2, 2

# Shape (B,T,H,D). Values are chosen only for transparent inspection.
q_bthd = mx.array([[[
    [1.0, 0.0],   # token 0, head 0
    [0.0, 1.0],   # token 0, head 1
], [
    [0.0, 1.0],   # token 1, head 0
    [1.0, 0.0],   # token 1, head 1
], [
    [1.0, 1.0],   # token 2, head 0
    [1.0, 1.0],   # token 2, head 1
]]])

q_bhtd = mx.swapaxes(q_bthd, 1, 2)

print('before:', q_bthd.shape)
print('after :', q_bhtd.shape)
print('q_bhtd:')
print(q_bhtd)

# Expected logical grouping after swap:
# head 0 -> [[1,0], [0,1], [1,1]]
# head 1 -> [[0,1], [1,0], [1,1]]
# Master notebook remains unexecuted.


## 3. Batched Matrix Multiplication：`(B,H)` 自动保留

现在：

$$
Q:(B,H,T,D)
$$

和：

$$
K:(B,H,T,D).
$$

为了做 Query–Key dot products，只交换 K 的最后两个 axes：

$$
K^T:(B,H,D,T).
$$

于是：

$$
\boxed{
(B,H,T,D)
@
(B,H,D,T)
\rightarrow
(B,H,T,T)
}
$$

在 MLX 0.32.2 中，`mx.matmul` 对多于二维的 arrays 做 batched matrix multiplication：

- 最后两个 axes 是 matrix dimensions；
- 更前面的 axes 按 broadcasting rules 作为 batch dimensions。

因此这里：

```text
B = batch dimension
H = another batch-like dimension
T,D × D,T = actual matrix product
```

结果：

$$
S[b,h,:,:]
\in
\mathbb R^{T\times T}
$$

就是 sample `b`、head `h` 自己的 attention score matrix。


## 4. Tiny Two-Head Example：一次 Matmul 得到两个独立 Score Matrices

继续使用：

```text
B = 1
H = 2
T = 3
D = 2
```

Head 0 的 Query rows：

$$
Q^{(0)}
=
\begin{bmatrix}
1&0\\
0&1\\
1&1
\end{bmatrix}.
$$

给 Head 0 选择：

$$
K^{(0)}
=
\begin{bmatrix}
1&0\\
1&1\\
0&1
\end{bmatrix}.
$$

所以：

$$
Q^{(0)}K^{(0)T}
=
\begin{bmatrix}
1&1&0\\
0&1&1\\
1&2&1
\end{bmatrix}.
$$

Head 1：

$$
Q^{(1)}
=
\begin{bmatrix}
0&1\\
1&0\\
1&1
\end{bmatrix},
$$

$$
K^{(1)}
=
\begin{bmatrix}
0&1\\
1&0\\
1&1
\end{bmatrix}.
$$

得到：

$$
Q^{(1)}K^{(1)T}
=
\begin{bmatrix}
1&0&1\\
0&1&1\\
1&1&2
\end{bmatrix}.
$$

所以一个 `(B,H,T,T)` tensor 可以同时保存：

```text
scores[0,0,:,:] -> Head 0 的 3×3 scores
scores[0,1,:,:] -> Head 1 的 3×3 scores
```

这就是“多个 heads 逻辑独立，但被一个 batched tensor operation 一次组织计算”的核心。


In [ ]:
# Q is the tensor created in the previous cell: q_bhtd -> (B,H,T,D).
q = q_bhtd

# Build K directly in (B,H,T,D) layout for a hand-checkable example.
k = mx.array([[
    [
        [1.0, 0.0],
        [1.0, 1.0],
        [0.0, 1.0],
    ],
    [
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ],
]])

k_t = mx.swapaxes(k, -1, -2)  # (B,H,D,T)
scores = mx.matmul(q, k_t)     # (B,H,T,T)

print('q shape:', q.shape)
print('k shape:', k.shape)
print('k_t shape:', k_t.shape)
print('scores shape:', scores.shape)
print('Head 0 scores:')
print(scores[0, 0])
print('Head 1 scores:')
print(scores[0, 1])

# Expected Head 0:
# [[1,1,0],
#  [0,1,1],
#  [1,2,1]]
# Expected Head 1:
# [[1,0,1],
#  [0,1,1],
#  [1,1,2]]
# This master notebook remains unexecuted.


## 5. Causal Mask 怎样扩展到 `(B,H,T,T)`？

Step 12 的 causal mask 仍然只依赖 Query/Key positions：

$$
M:(T,T).
$$

现在 scores：

$$
S:(B,H,T,T).
$$

相加时 broadcasting：

$$
\boxed{
(B,H,T,T)
+
(T,T)
\rightarrow
(B,H,T,T)
}
$$

所以同一个 causal visibility pattern 可以同时应用到：

- 所有 batch samples；
- 所有 heads。

而每个 `(b,h)` 的实际 compatibility values 仍然来自自己对应的 Q/K channels。

也就是说：

```text
mask structure shared
attention scores head-specific
```

本节可以理解这个 broadcasting 关系，但不需要重新展开 Step 12 已经学过的 stable-softmax 推导。


In [ ]:
T = scores.shape[-1]

future = mx.triu(mx.ones((T, T)), k=1) > 0
mask = mx.where(
    future,
    mx.array(float('-inf'), dtype=scores.dtype),
    mx.array(0.0, dtype=scores.dtype),
)

masked_scores = scores + mask  # (B,H,T,T) + (T,T)

print('mask shape:', mask.shape)
print('masked_scores shape:', masked_scores.shape)

# No head concatenation or output projection in Step 13.4.


## 6. “Heads 独立”到底是什么意思？

在当前计算里：

$$
S[b,h]
=
Q[b,h]K[b,h]^T.
$$

所以 Head 0 的 score matrix 只使用：

```text
Q[:,0,:,:]
K[:,0,:,:]
```

Head 1 只使用：

```text
Q[:,1,:,:]
K[:,1,:,:]
```

没有做：

$$
Q^{(0)}K^{(1)T}
$$

这种 cross-head dot product。

这就是本节所谓 **logical head independence（逻辑上的 head 独立）**：

> 在 head-wise attention score computation 中，每个 head 使用自己的 D-dimensional Q/K channels 形成自己的 T×T attention problem。

但这种独立是 tensor computation 的逻辑结构，不意味着 heads 在整个网络中永远互不影响；后续 architecture 可以重新组合多个 head outputs。那属于 Step 13.5。


## 7. 不要把它理解成“一 Head = 一个 GPU Core”

看到：

$$
(B,H,T,D)
$$

很容易产生一种过度字面化理解：

```text
Head 0 -> GPU core 0
Head 1 -> GPU core 1
...
```

这不是正确的硬件模型。

代码提交的是一个 batched tensor operation，例如：

```python
mx.matmul(q, k_t)
```

具体怎样把：

- batches；
- heads；
- T×D matrix tiles；
- arithmetic work

调度到 Apple GPU 的 execution resources，是 MLX / compiler / GPU kernel implementation 的事情。

一个 head 的矩阵运算通常会涉及大量并行 execution lanes / hardware resources；同样，一个 hardware core 也不是固定只服务某个 head。

因此准确区分：

$$
\boxed{
\text{logical head axis}
\neq
\text{physical GPU-core mapping}
}
$$

我们现在只学习 tensor algebra，不给 H 个 logical heads 指派 H 个物理 GPU cores。


## 8. Step 13.4 材料边界

- Step 13.3 的 `(B,T,H,D)` 通过交换 T/H axes 变成 `(B,H,T,D)`；精确关系是 `Q_BHTD[b,h,t,d] = Q_BTHD[b,t,h,d]`。
- `(B,H)` 作为 batched matmul 的 batch-like dimensions；最后两个 axes `(T,D)` 才是每个 head 的矩阵。
- K 再交换最后两个 axes得到 `(B,H,D,T)`，所以 `mx.matmul(Q,K^T)` 得到 `(B,H,T,T)`。
- `scores[b,h,:,:]` 是 sample b、head h 自己的 T×T compatibility matrix；head-wise score computation 不包含 cross-head Q/K dot products。
- `(T,T)` causal mask 可以 broadcast 到 `(B,H,T,T)`，同一个 visibility rule 作用于全部 batch samples 与 heads。
- logical head independence 不等于“一 head 一个 GPU core”；physical scheduling 由 MLX/compiler/kernel/hardware 决定。
- 本轮只准备 Step 13.4；不进入 Step 13.5，不把 head outputs transpose back，不 reshape/concatenate 成 `(B,T,C)`，也不引入 `c_proj` / output projection。
